# Training of Noncoupled Nonlinear Memristor Models

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import random

from functions import *

## Simple Schottky-Tunneling Current

### Same Input & Parameters, Different Initial Conditions

In [ ]:
T = 50
dt = 1e-4
t = np.arange(0, T, dt)

N = 100

nu = 1
v_amp = 1/2
v_input = np.sin(2 * np.pi * nu * t)
#v_input = v_amp*(np.sin(2 * np.pi * t * (1/nu)) + np.sin(5 * np.pi * t * (1/nu)))

# Memristor model parameters from Singh et al paper
alpha, beta, gamma, delta = 2.5e-6, 0.5, 2.5e-6, 4
lam, eta, tau = 2.5, 3, 0.05


#alpha, beta, gamma, delta = 6.8299149, 0.10051066, 1.6138e-04, 0.99905798
#lam, eta, tau = 3.05688105, 0.19781416, 0.10820846

#alpha, beta, gamma, delta = 1e-7, 1e-3, 1e-7, 1e-2
#lam, eta, tau = 4.44, 6.67e-1, 4.45e-1

#alpha, beta, gamma, delta = 1e-8, 5e-1, 1e-5, 4
#lam, eta, tau = 1e-3, 8, 5e-2


params = [alpha, beta, gamma, delta, lam, eta, tau]
pert_step = 0
max_pert = 0
idx_array = [i for i in range(len(params))]
'''
    x = np.random.rand(N)  # initial states for each memristor
    #x = 0
    #x = 0.9*np.ones(N)
'''

param_array = param_network(N,params, pert_step, max_pert, idx_array)
print(param_array.shape)

output, X = simulate_memristor_network_current(N, t, v_input, param_array)
print(output.shape,X.shape)



In [ ]:
# Train
training_time = 30

#sig = v_input*v_input
sig = np.tanh(v_input)
sig = np.expand_dims(sig,axis=1)
sig = (sig-np.mean(sig))/np.std(sig)
sup = sig[:int(training_time/dt),:]


lam_d = 1e-10
basis = output[:int(training_time/dt),:]
print(output.shape)
print(basis.shape,sup.shape)

phi = np.linalg.pinv(basis.T@basis+lam_d*np.eye(len(output[0])))@(basis.T@sup)
print(phi.shape)
# Predict and evaluate
prediction = output@phi

In [ ]:
# Plotting the target and prediction
plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Target', color='blue')
plt.plot(t, prediction, label='Supervisor', color='orange', linestyle='--')
plt.xlabel("Time (s)")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 4))
plt.plot(t[round(training_time/dt):], sig[round(training_time/dt):], label='Supervisor', color='blue')
plt.plot(t[round(training_time/dt):], prediction[round(training_time/dt):], label='Prediction', color='red', linestyle='--')
plt.xlabel("Time (s)")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
plt.grid(True, linestyle="--", alpha=0.4)
plt.ylim(1.1*np.min(sig[round(training_time/dt):]),1.1*np.max(sig[round(training_time/dt):]))
plt.xlim(training_time,T)
plt.tight_layout()
plt.savefig('Committee Meeting Plots/same_params_diff_y0.png', dpi=300, bbox_inches="tight")
plt.show()

rmse = np.sqrt(np.mean((prediction[t>training_time] - sig[t>training_time])**2))

print(f"RMSE: {rmse:.4e}")


plt.figure(figsize=(10,4))
for curr in output.T:
    plt.plot(t[:round(5/dt)],curr[:round(5/dt)])
plt.ylabel('Current')
plt.xlabel('Time')
plt.title('Current Traces')
plt.show()
plt.figure()
plt.plot(t,X)
plt.ylabel('State')
plt.xlabel('Time')
plt.title('State Dynamics')
plt.show()


plt.figure(figsize=(10,10))
plt.plot(v_input,output.T[10])
plt.title('I-V Graph')
plt.xlabel('Voltage')
plt.ylabel('Current')
plt.show()

In [ ]:
labels = ["α","β","γ","δ","λ","η","τ"]

for i in range(len(params)):
    print(f"Max {labels[i]} value:", max(param_array[i]), f"\nMin {labels[i]} value:", 
          min(param_array[i]), f"\nOriginal {labels[i]} value:", params[i])
    plt.figure()
    plt.boxplot(param_array[i])
    plt.title(f"Parameter {labels[i]}")
    plt.axhline(y=params[i], color='r', linestyle='--', label='Original Value')
    plt.legend()
    plt.show()

### Same Input, Different Initial Conditions & Parameters

In [ ]:
T = 50
dt = 1e-4
t = np.arange(0, T, dt)

N = 300

nu = 1
v_amp = 1/2
#v_input = np.sin(2 * np.pi * nu * t)
v_input = v_amp*(np.sin(2 * np.pi * t * (1/nu)) + np.sin(5 * np.pi * t * (1/nu)))

# Memristor model parameters
alpha, beta, gamma, delta = 2.5e-6, 0.5, 2.5e-6, 4
lam, eta, tau = 2.5, 3, 0.05


#alpha, beta, gamma, delta = 6.8299149, 0.10051066, 1.6138e-04, 0.99905798
#lam, eta, tau = 3.05688105, 0.19781416, 0.10820846

#alpha, beta, gamma, delta = 1e-7, 1e-3, 1e-7, 1e-2
#lam, eta, tau = 4.44, 6.67e-1, 4.45e-1

#alpha, beta, gamma, delta = 1e-8, 5e-1, 1e-5, 4
#lam, eta, tau = 1e-3, 8, 5e-2


params = [alpha, beta, gamma, delta, lam, eta, tau]
pert_step = 5
max_pert = 10
idx_array = [i for i in range(len(params))]

param_array = param_network(N,params, pert_step, max_pert, idx_array)
#print(param_array.shape)
param_array_normal = param_normal_network(N,params,1e-1)
print(param_array_normal.shape)


output, X = simulate_memristor_network_current(N, t, v_input, param_array_normal)
#print(output.shape,X.shape)

In [ ]:
# Train
training_time = 20

sig = v_input*v_input
#sig = np.tanh(v_input)
sig = np.expand_dims(sig,axis=1)
sig = (sig-np.mean(sig))/np.std(sig)
sup = sig[:int(training_time/dt),:]


lam_d = 1e-10
basis = output[:int(training_time/dt),:]
print(output.shape)
print(basis.shape,sup.shape)

phi = np.linalg.pinv(basis.T@basis+lam_d*np.eye(len(output[0])))@(basis.T@sup)
print(phi.shape)
# Predict and evaluate
prediction = output@phi

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Supervisor', color='blue')
plt.plot(t, prediction, label='Prediction', color='red', linestyle='--')
plt.xlabel("Time (s)")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.tight_layout()
plt.xlim(training_time,training_time+20)
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
#plt.grid(True)
plt.grid(True, linestyle="--", alpha=0.4)
plt.ylim(1.1*np.min(sig),1.1*np.max(sig))
plt.show()


plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Target', color='blue')
plt.plot(t, prediction, label='Supervisor', color='orange', linestyle='--')
plt.xlabel("Time")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.xlim(0,training_time)
plt.show()

mse = np.mean((prediction[t>training_time] - sig[t>training_time]))

print(f"MSE: {mse:.4e}")

plt.figure()
for curr in output.T:
    plt.plot(t[:round(3/dt)],curr[:round(3/dt)])
plt.ylabel('Current (A)')
plt.xlabel('Time (s)')
plt.title('Current Traces')
plt.show()


plt.figure()
plt.plot(t,X)
plt.ylabel('State')
plt.xlabel('Time')
plt.title('State Dynamics')
plt.show()

print('Ouput shape:',np.shape(output),'\n t shape:',np.shape(t))

In [ ]:
labels = ["α","β","γ","δ","λ","η","τ"]

for i in range(len(params)):
    print(f"Max {labels[i]} value:", max(param_array_normal[i]), f"\nMin {labels[i]} value:", 
          min(param_array_normal[i]), f"\nOriginal {labels[i]} value:", params[i])
    plt.figure()
    plt.boxplot(param_array_normal[i])
    plt.title(f"Parameter {labels[i]}")
    plt.axhline(y=params[i], color='r', linestyle='--', label='Original Value')
    plt.legend()
    plt.show()


###  Input Delays, Different Initial Conditions & Parameters

In [ ]:
T = 50
dt = 1e-4
t = np.arange(0, T, dt)

N = 100

nu = 1.5    # 1.5 for added sinusoids
v_amp = 1/2
#v_input = np.sin(2 * np.pi * nu * t)
v_input = v_amp*(np.sin(2 * np.pi * t) + np.sin(5 * np.pi * t))

v_list_shift, shift_list = shift(t,v_input,nu,N)

# Memristor model parameters
alpha, beta, gamma, delta = 2.5e-6, 0.5, 2.5e-6, 4
lam, eta, tau = 2.5, 3, 0.05


#alpha, beta, gamma, delta = 6.8299149, 0.10051066, 1.6138e-04, 0.99905798
#lam, eta, tau = 3.05688105, 0.19781416, 0.10820846

#alpha, beta, gamma, delta = 1e-7, 1e-3, 1e-7, 1e-2
#lam, eta, tau = 4.44, 6.67e-1, 4.45e-1

#alpha, beta, gamma, delta = 1e-8, 5e-1, 1e-5, 4
#lam, eta, tau = 1e-3, 8, 5e-2


params = [alpha, beta, gamma, delta, lam, eta, tau]
pert_step = 5
max_pert = 10
idx_array = [i for i in range(len(params))]

param_array = param_network(N,params, pert_step, max_pert, idx_array)
print(param_array.shape)
param_array_normal = param_normal_network(N,params,1e-1)
print(param_array_normal.shape)

output, X = simulate_memristor_network_current(N, t, v_list_shift, param_array_normal)
#print(output.shape,X.shape)

In [ ]:
# Train
training_time = 20

sig = v_input*v_input
#sig = np.tanh(v_input)
sig = np.expand_dims(sig,axis=1)
sig = (sig-np.mean(sig))/np.std(sig)
sup = sig[:int(training_time/dt),:]


lam_d = 1e-10
basis = output[:int(training_time/dt),:]
print(output.shape)
print(basis.shape,sup.shape)

phi = np.linalg.pinv(basis.T@basis+lam_d*np.eye(len(output[0])))@(basis.T@sup)
print(phi.shape)
# Predict and evaluate
prediction = output@phi

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Supervisor', color='blue')
plt.plot(t, prediction, label='Prediction', color='red', linestyle='--')
plt.xlabel("Time (s)")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.tight_layout()
plt.xlim(training_time,35)
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
#plt.grid(True)
plt.grid(True, linestyle="--", alpha=0.4)
plt.ylim(1.1*np.min(sig),1.1*np.max(sig))
#plt.savefig('Committee Meeting Plots/inp_delay_diff_params_diff_y0.png', dpi=300, bbox_inches="tight")
plt.savefig('Committee Meeting Plots/complex_inp_delay_diff_params_diff_y0.png', dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Target', color='blue')
plt.plot(t, prediction, label='Supervisor', color='orange', linestyle='--')
plt.xlabel("Time")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.xlim(0,training_time)
plt.show()

mse = np.mean((prediction[t>training_time] - sig[t>training_time]**2))

print(f"MSE: {mse:.4e}")


plt.figure()
for curr in output.T:
    plt.plot(t[:round(3/dt)],curr[:round(3/dt)])
plt.ylabel('Current')
plt.xlabel('Time')
plt.title('Current Traces')
plt.show()


plt.figure()
plt.plot(t,X)
plt.ylabel('State')
plt.xlabel('Time')
plt.title('State Dynamics')
plt.show()

print('Ouput shape:',np.shape(output),'\n t shape:',np.shape(t))

In [ ]:
labels = ["α","β","γ","δ","λ","η","τ"]

for i in range(len(params)):
    print(f"Max {labels[i]} value:", max(param_array_normal[i]), f"\nMin {labels[i]} value:", 
          min(param_array_normal[i]), f"\nOriginal {labels[i]} value:", params[i])
    plt.figure()
    plt.boxplot(param_array_normal[i])
    plt.title(f"Parameter {labels[i]}")
    plt.axhline(y=params[i], color='r', linestyle='--', label='Original Value')
    plt.legend()
    plt.show()

## Using the same functions and parameters in Claudia's paper (MM1-$\tau$ w/ recitifier current)

In [ ]:
#α1, β1, α2, β2, γ, δ = 6.99172421, 0.09768957, 5.69422726, 0.01078093, 1.5903e-4, 1.00130584
#λ1, λ2, η1, η2, η3, η4, τ =  3.31522859, 3.79686517, 0.18971782, 0.38420470, 0.18220223, 0.49678829, 0.1109234

α1, β1, α2, β2, γ, δ = 6.89172421, 0.09768957 ,3.3451944, 0.010780930, 1.6138e-04, 0.99905798
λ1, λ2, η1, η2, η3, η4, τ =  3.31522859, 3.79686517, 0.18971782, 0.38420470, 0.18220223, 0.49678829, 0.21092344


#α1, β1, α2, β2, γ, δ = 8.345194, 0.1078093, 3.3451944, 0.01078093, 0.005, 1.039012
#λ1, λ2, η1, η2, η3, η4, τ =  3.254168, 3.254168, 0.26103, 0.3249405, 0.26103, 0.3249405, 0.03998672

T = 50
dt = 1e-4
t = np.arange(0, T, dt)
nu = 1
v_amp = 1
v = v_amp*np.sin(2 * np.pi * nu * t)
#print('size of v[1]', v.shape[0])
sig = np.tanh(v)
#sig = (sig - np.mean(sig))/np.std(sig)

N = 100

curr_params = [α1, β1, α2, β2, γ, δ]
state_params = [λ1, λ2, η1, η2, η3, η4, τ]

pert_step = 5e-2
max_pert = 10
curr_idx_array = [i for i in range(len(curr_params))]
state_idx_array = [i for i in range(len(state_params))]

curr_params_network = param_network(N, curr_params, pert_step, max_pert, curr_idx_array)
state_params_network = param_network(N, state_params, pert_step, max_pert, state_idx_array)


# Generate input-output data
output, store_x = current_mm1tau(t, N, v, curr_params_network, state_params_network)

training_time = 30

sig = v*v
sig = np.expand_dims(sig,axis=1)
sig=(sig-np.mean(sig))/np.std(sig)
sup = sig[:int(30/dt),:]



lam_d = 1e-10
basis = output[:,:int(30/dt)].T
print(basis.shape,sup.shape)


phi = np.linalg.pinv(basis.T@basis+lam_d*np.eye(len(output)))@(basis.T@sup)

# Predict and evaluate
prediction = phi.T @ output
print(prediction.shape)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t[int(training_time/dt):], sig[int(training_time/dt):], label='Target', color='blue')
plt.plot(t[int(training_time/dt):], prediction.T[int(training_time/dt):], label='Prediction', color='orange', linestyle='--')
plt.xlabel("Time")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

mse = np.mean((prediction.T[t>training_time] - sig[t>training_time]**2))

print(f"MSE: {mse:.4e}")

plt.figure()
plt.plot(t,output.T)
plt.ylabel('Current')
plt.xlabel('Time')
plt.title('Current Traces')
plt.show()

plt.figure()
plt.plot(v, output.T)
plt.show()
plt.figure()

plt.figure()
for curr in output.T:
    plt.plot(t[:round(1/dt)],curr[:round(1/dt)])
plt.ylabel('Current')
plt.xlabel('Time')
plt.title('Current Traces')
plt.show()

plt.figure()
plt.plot(v, output.T[1])
plt.show()
plt.figure()

plt.plot(t,store_x)
plt.ylabel('State')
plt.xlabel('Time')
plt.title('State Dynamics')

plt.show()


In [ ]:
#α1, β1, α2, β2, γ, δ = 6.99172421, 0.09768957, 5.69422726, 0.01078093, 1.5903e-4, 1.00130584
#λ1, λ2, η1, η2, η3, η4, τ =  3.31522859, 3.79686517, 0.18971782, 0.38420470, 0.18220223, 0.49678829, 0.1109234

α1, β1, α2, β2, γ, δ = 6.89172421, 0.09768957, 3.3451944, 0.010780930, 1.6138e-04, 0.99905798
λ1, λ2, η1, η2, η3, η4, τ =  3.31522859, 3.79686517, 0.18971782, 0.38420470, 0.18220223, 0.49678829, 0.21092344

#α1, β1, α2, β2, γ, δ = 8.345194, 0.1078093, 3.3451944, 0.01078093, 0.005, 1.039012
#λ1, λ2, η1, η2, η3, η4, τ =  3.254168, 3.254168, 0.26103, 0.3249405, 0.26103, 0.3249405, 0.03998672

T = 50
dt = 1e-4
t = np.arange(0, T, dt)
nu = 1

N = 400

v = np.sin(2 * np.pi * t * (1/nu)) + np.sin(5 * np.pi * t * (1/nu))

v_list_shift, shift_list = shift(t,v,nu,N)

v_list_phase = [np.sin(2*np.pi*(t+np.random.random())*nu) for _ in range(N)]


curr_params = [α1, β1, α2, β2, γ, δ]
state_params = [λ1, λ2, η1, η2, η3, η4, τ]

pert_step = 5e-2
max_pert = 10
curr_idx_array = [i for i in range(len(curr_params))]
state_idx_array = [i for i in range(len(state_params))]

curr_params_network = param_network(N, curr_params, pert_step, max_pert, curr_idx_array)
state_params_network = param_network(N, state_params, pert_step, max_pert, state_idx_array)

# Generate input-output data
#print(v_list_shift.shape,curr_params_network.shape)
output, store_x = current_mm1tau(t, N, v_list_shift, curr_params_network, state_params_network)
#output, store_x = current_mm1tau(t, N, v, curr_params_network, state_params_network)

In [ ]:
training_time = 30

sig = v*v
sig = np.expand_dims(sig,axis=1)
sig=(sig-np.mean(sig))/np.std(sig)
sup = sig[:int(30/dt),:]


lam_d = 1e-10
basis = output[:,:int(30/dt)].T
print(basis.shape,sup.shape)

phi = np.linalg.pinv(basis.T@basis+lam_d*np.eye(len(output)))@(basis.T@sup)
print(phi.shape)
# Predict and evaluate
prediction = phi.T@output

In [ ]:
print(sig.shape,prediction.shape)
plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Supervisor', color='blue')
plt.plot(t, prediction.T, label='Prediction', color='red', linestyle='--')
plt.xlabel("Time")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.xlim(training_time,40)
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
#plt.grid(True)
plt.grid(True, linestyle="--", alpha=0.4)
plt.ylim(1.1*np.min(sig),1.1*np.max(sig))
#plt.savefig('Committee Meeting Plots/complex_sig_shift_10mem.png', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
labels = ["α1", "β1", "α2", "β2", "γ","δ"]

for i in range(len(curr_params)):
    print(f"Max {labels[i]} value:", max(curr_params_network.T[:,i]), f"\nMin {labels[i]} value:", 
          min(curr_params_network.T[:,i]), f"\nOriginal {labels[i]} value:", curr_params[i])
    plt.figure()
    plt.boxplot(curr_params_network.T[:,i])
    plt.title(f"Parameter {labels[i]}")
    plt.axhline(y=curr_params[i], color='r', linestyle='--', label='Original Value')
    plt.legend()
    plt.show()